# SDUM 어댑터 검증 (Colab, 옵션 A)

이 노트북은 **내 컴퓨터에 아무것도 설치하지 않고** Colab(무료 GPU)에서 SDUM 이식 코드가 실제로 도는지 검증한다.

## 먼저 할 것 (한 번)
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)** 선택 → 저장.
2. **셀을 위에서부터 순서대로 실행**(Shift+Enter) 또는 **런타임 → 모두 실행**.

## 코드 가져오는 방법 두 가지 (셀 3-A 또는 3-B 중 하나만)
- **A. GitHub clone** (repo가 private면 Personal Access Token 필요): 셀 3-A 실행.
- **B. zip 업로드** (토큰 없이): 로컬에서 `FastMRI_SNU_2026_GMG` 폴더를 zip으로 압축해 두고 셀 3-B 실행 후 업로드.


## 셀 1 — GPU / torch 확인

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 셀 2 — 의존성 설치
Colab엔 torch가 이미 있으니 MONAI/einops/timm/pytest만 설치(빠름).

In [ ]:
!pip install -q monai einops timm pytest

## 셀 3-A — 코드 가져오기 (GitHub clone)
private repo면 실행 시 토큰 입력창이 뜬다. GitHub → Settings → Developer settings → Personal access tokens에서 **repo 권한** 토큰 발급.
(public repo면 토큰 없이 그냥 Enter.)

In [ ]:
import getpass, os
USER, REPO, BRANCH = 'JunKyungSeo', 'FastMRI_SNU_2026_GMG', 'SDUM'
tok = getpass.getpass('GitHub Personal Access Token (public이면 그냥 Enter): ').strip()
auth = f'{tok}@' if tok else ''
url = f'https://{auth}github.com/{USER}/{REPO}.git'
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {url}
%cd {REPO}
!git log --oneline -3

## 셀 3-B — (대안) zip 업로드로 코드 가져오기
셀 3-A를 썼으면 이 셀은 건너뛴다. 로컬 `FastMRI_SNU_2026_GMG` 폴더를 zip으로 만들어 업로드.

In [ ]:
from google.colab import files
import zipfile, glob, os
up = files.upload()   # FastMRI_SNU_2026_GMG.zip 선택
zp = list(up)[0]
with zipfile.ZipFile(zp) as z:
    z.extractall('.')
# 압축 안에 repo 폴더가 있으면 그 안으로, 아니면 현재 폴더 사용
cand = [d for d in glob.glob('**/utils/model/sdum', recursive=True)]
root = os.path.dirname(os.path.dirname(os.path.dirname(cand[0]))) if cand else '.'
os.chdir(root or '.')
print('repo root:', os.getcwd())
print(os.listdir('.'))

## 셀 4 — ⭐ 어댑터 검증 (핵심)
`python -m pytest`로 실행(현재 폴더를 import 경로에 추가하기 위함). 여기서 초록불이면 SDUM 이식이 실제로 도는 것.

In [ ]:
!python -m pytest tests/ -v

## 셀 5 — (GPU) 실제 메모리 탐침
Colab는 T4(≈15GB)라 1080 8GB는 아니지만, **GPU에서 도는지 + 대략 peak 메모리**를 확인. 8GB 목표는 나중에 실제 1080/VESSL에서 재확인.

In [ ]:
!python scripts/probe_oom.py --cascades 4 --channels 32 64
!python scripts/probe_oom.py --cascades 6 --channels 24 48